In [1]:
import sys
from pathlib import Path

import pandas as pd
import numpy as np
import joblib

ROOT_DIR = Path.cwd().parent

sys.path.insert(
    0,
    str(ROOT_DIR)
)

PROCESSED_DIR = (
    ROOT_DIR /
    "data" /
    "processed"
)

MODEL_DIR = (
    ROOT_DIR /
    "models"
)

print("Project:", ROOT_DIR)

Project: c:\Users\SHUBHAM\OneDrive\Desktop\AlphaTrade


In [2]:
def predict_stock(ticker):

    data_file = (
        PROCESSED_DIR /
        f"{ticker}_features.csv"
    )

    model_file = (
        MODEL_DIR /
        f"{ticker}_model.pkl"
    )

    feature_file = (
        MODEL_DIR /
        f"{ticker}_feature_columns.pkl"
    )

    if not data_file.exists():
        raise FileNotFoundError(
            f"Data not found: {data_file}"
        )

    if not model_file.exists():
        raise FileNotFoundError(
            f"Model not found: {model_file}"
        )

    # Load data
    df = pd.read_csv(
        data_file
    )

    # Load model
    model = joblib.load(
        model_file
    )

    # Load feature columns
    feature_columns = joblib.load(
        feature_file
    )

    # Latest row
    latest = df.iloc[-1]

    X_latest = df[
        feature_columns
    ].iloc[[-1]]

    # Prediction
    predicted_price = model.predict(
        X_latest
    )[0]

    current_price = latest["Close"]

    change_percent = (
        (predicted_price - current_price)
        / current_price
    ) * 100

    # Signal
    if change_percent >= 2:
        signal = "BUY"
    elif change_percent <= -2:
        signal = "SELL"
    else:
        signal = "HOLD"

    return {
        "Ticker": ticker,
        "Current Price": current_price,
        "Predicted Price": predicted_price,
        "Expected Change (%)": change_percent,
        "Signal": signal
    }

In [3]:
result = predict_stock("AAPL")

result

{'Ticker': 'AAPL',
 'Current Price': np.float64(305.260009765625),
 'Predicted Price': np.float64(195.3053000485768),
 'Expected Change (%)': np.float64(-36.02001775518192),
 'Signal': 'SELL'}

In [4]:
TICKERS = [
    "AAPL",
    "MSFT",
    "GOOGL",
    "AMZN",
    "TSLA",
    "NVDA"
]

predictions = []

for ticker in TICKERS:

    result = predict_stock(
        ticker
    )

    predictions.append(
        result
    )

predictions_df = pd.DataFrame(
    predictions
)

predictions_df

,Ticker,Current Price,Predicted Price,Expected Change (%),Signal
0,AAPL,305.260010,195.305300,-36.020018,SELL
1,MSFT,496.880005,421.902483,-15.089664,SELL
2,GOOGL,346.359985,162.327672,-53.133249,SELL
3,AMZN,265.130005,184.391694,-30.452348,SELL
4,TSLA,339.959991,339.366486,-0.174581,HOLD
5,NVDA,225.300003,89.500403,-60.275010,SELL


In [5]:
predictions_df.to_csv(
    PROCESSED_DIR /
    "latest_predictions.csv",
    index=False
)

print("✅ Predictions saved")

✅ Predictions saved
